<div style='background:linear-gradient(135deg,#1A2E4A 0%,#0D7377 100%);padding:50px 40px;border-radius:12px;color:white;text-align:center;font-family:Arial,sans-serif;'>
  <p style='font-size:13px;letter-spacing:3px;color:#14BDBD;margin:0 0 8px 0;'>AI / ML FOUNDATIONS COHORT — CAPSTONE PROJECT 2</p>
  <h1 style='font-size:38px;margin:0 0 8px 0;font-weight:900;'>House Price Prediction</h1>
  <h2 style='font-size:22px;font-weight:300;margin:0 0 10px 0;color:#D0D7E3;'>& Personalised House Recommender System</h2>
  <p style='font-size:14px;color:#F0A500;font-weight:bold;margin:0 0 30px 0;'>End-to-End ML Project + Streamlit Deployment</p>
  <div style='width:60px;height:3px;background:#F0A500;margin:0 auto 30px auto;'></div>
  <p style='font-size:14px;color:#D0D7E3;margin:0 0 6px 0;'>Your data. Your model. Your deployment.</p>
  <p style='font-size:13px;color:#6B8A9A;margin:0;'>This is a project brief — not a tutorial. You design, build, and ship it.</p>
</div>


---
## 📋 What This Notebook Is

This is your **project brief and architectural guide** — not a walkthrough with code handed to you.

You will:
- Source your own dataset
- Make your own modelling decisions
- Build two interconnected systems: a **price predictor** and a **house recommender**
- Deploy both as a live web application using **Streamlit**

This notebook gives you:
- A clear problem statement and success criteria
- Architectural guidance and design options
- Code scaffolds and key hints (not full solutions)
- A deployment guide for Streamlit
- An evaluation rubric

> 🔑 **The gap between a student who follows tutorials and a practitioner who builds  
> real things is crossed exactly here. This is your crossing.**


---

# 🏠 Section 1: Project Overview
#### *What you are building and why it matters*

---

### The Real-World Problem

Real estate is one of the most data-rich, high-stakes domains in the world.  
A buyer wants to know: **Is this house priced fairly?**  
An agent wants to know: **What other houses should I show this client?**  
A developer wants to know: **What features add the most value in this market?**

All three questions are ML problems. You will build systems that answer the first two.

---

### What You Are Building

**System 1 — Price Predictor**  
A regression model that takes house features as input and outputs a predicted price.  
The user enters: bedrooms, bathrooms, size, location, age, amenities, etc.  
The model outputs: a price estimate with a confidence range.

**System 2 — House Recommender**  
Given the house a user described (to get a price), find and recommend the most similar  
houses in your dataset. The idea: the user has expressed a preference by describing  
their ideal house — now surface real alternatives they might buy.

**Deployment — Streamlit App**  
Both systems live in a single Streamlit web app with a clean, professional UI.  
Users interact with sliders, dropdowns, and number inputs.  
They get a price prediction AND a list of recommended similar properties.

---

### Why This Project Is Valuable

- It combines regression + recommendation — two of the most common ML task types
- It requires the full pipeline: EDA → cleaning → feature engineering → modelling → deployment
- It produces a real, usable, demonstrable product
- It is highly relevant across African real estate markets (Lagos, Nairobi, Accra, Johannesburg)
  where pricing is often opaque and agents have limited analytical tools
- It is portfolio-ready — a recruiter or client can use your app directly


---

# 📦 Section 2: Sourcing Your Dataset
#### *Your choice — your market — your domain knowledge*

---

### You Must Source Your Own Data

No dataset is provided. This is intentional. Part of the project is demonstrating  
that you can find, evaluate, and work with data independently.

### Dataset Requirements

| Requirement | Minimum |
|-------------|--------|
| Rows | 500+ records |
| Target column | A numeric house/property price |
| Feature types | Mix of numeric and categorical |
| Numeric features | At least 5 (size, bedrooms, bathrooms, age, etc.) |
| Categorical features | At least 3 (location, type, condition, etc.) |

### Where to Find Data

**Option A — Public Datasets (quickest start):**
- Ames Housing Dataset (Kaggle) — classic, 79 features, 2919 samples
- California Housing Dataset — sklearn built-in (`from sklearn.datasets import fetch_california_housing`)
- King County, WA House Sales (Kaggle) — 21,613 houses with GPS coordinates
- Melbourne Housing Snapshot (Kaggle)
- Nigeria/Ghana/Kenya real estate scrapes available on Kaggle and GitHub

**Option B — Scrape It Yourself (harder, more impressive):**
- PropertyPro.ng, PrivateProperty.com.ng (Nigeria)
- BuyRentKenya.com (Kenya)
- Meqasa.com (Ghana)
- Property24.com (South Africa)
- Use `requests` + `BeautifulSoup` or `Scrapy`
- Respect robots.txt and rate limits

**Option C — Hybrid:**
Use a public dataset but enrich it with data from another source  
(e.g. add neighbourhood crime rates, school ratings, distance to amenities via APIs)

### What Makes a Better Dataset
- More rows → better models, more interesting recommendations
- Local market data → more relevant, more unique portfolio piece
- Richer features → more interesting feature engineering
- Geographic data (lat/lon or neighbourhood) → enables map visualisations


In [ ]:
# ── Your data loading cell — replace with your actual data ───────────────────
import pandas as pd
import numpy as np

# Option A: Load from CSV
# df = pd.read_csv('your_housing_data.csv')

# Option B: sklearn California Housing (as a quick start placeholder)
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(as_frame=True)
df = housing.frame
df.rename(columns={'MedHouseVal': 'price'}, inplace=True)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Price range: ${df["price"].min():.2f} – ${df["price"].max():.2f}')
df.head()
# NOTE: Replace this with YOUR dataset before building the full project


---

# 🗺️ Section 3: The Full ML Pipeline — Your Roadmap
#### *Steps to follow, decisions to make*

---

### Step 1 — Exploratory Data Analysis

Before building anything, spend serious time understanding your data.  
Answer these questions with code and visualisations:

- What is the distribution of house prices? Is it skewed? Should you log-transform it?
- Which features correlate most strongly with price?
- Are there missing values? Where and how many?
- Are there outliers in price or features? Are they errors or genuine luxury properties?
- How many unique values do categorical features have?
- Is there geographic variation? Do prices differ significantly by neighbourhood/city?

**Minimum EDA outputs:**
- Price distribution histogram (raw and log-transformed)
- Correlation heatmap with price
- Missing value summary table
- Box plots: price by key categorical features
- Scatter plots: price vs top numeric features


In [ ]:
# ── EDA scaffold — build on this ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Price distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['price'].hist(bins=50, ax=axes[0], color='#0D7377', edgecolor='white')
axes[0].set_title('Price Distribution (Raw)')
np.log1p(df['price']).hist(bins=50, ax=axes[1], color='#F0A500', edgecolor='white')
axes[1].set_title('Price Distribution (Log-Transformed)')
plt.tight_layout(); plt.show()

# 2. Missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print('Missing values:')
    print(missing)
else:
    print('No missing values detected.')

# 3. Correlation with price
numeric_df = df.select_dtypes(include=np.number)
corr_with_price = numeric_df.corr()['price'].drop('price').sort_values(key=abs, ascending=False)
print('\nTop correlations with price:')
print(corr_with_price.head(10))

# Continue with more EDA...
# Add your own charts here based on your specific dataset


### Step 2 — Data Cleaning

Based on your EDA findings, clean the data. Document every decision.

**Checklist:**
- [ ] Handle missing values — justify each strategy (drop vs impute)
- [ ] Cap or remove price outliers — define your upper/lower thresholds
- [ ] Correct data types — ensure numerics are not stored as strings
- [ ] Remove duplicates
- [ ] Validate ranges — age cannot be negative, bedrooms cannot be 0 or 50

### Step 3 — Feature Engineering

This is where you create value. Good features can matter more than the model choice.

**Suggested features to create:**
- `price_per_sqm` = price / total_area (useful for recommendations)
- `age_group` = bin property age into categories (New, Modern, Established, Old)
- `total_rooms` = bedrooms + bathrooms + other rooms
- `has_garage` / `has_pool` / `has_garden` — binary flags from descriptive columns
- `neighbourhood_avg_price` = mean price per neighbourhood (powerful signal)
- `size_to_rooms_ratio` = total area / total rooms
- If you have datetime data: `year_sold`, `month_sold`, `days_on_market`
- If you have GPS: `distance_to_city_centre` using Haversine formula

### Step 4 — Preprocessing for Modelling

Build a scikit-learn Pipeline that handles:
- Numeric features: impute → scale (StandardScaler or RobustScaler)
- Categorical features: impute → one-hot encode or ordinal encode
- Target: consider log-transforming price (log1p), then inverse-transform predictions (expm1)


In [ ]:
# ── Preprocessing pipeline scaffold ──────────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

# Define your column types — update to match YOUR dataset
numeric_features     = []  # e.g. ['bedrooms','bathrooms','sqm','age']
categorical_features = []  # e.g. ['neighbourhood','property_type','condition']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler())   # robust to outliers in housing data
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,     numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Log-transform the target
# y = np.log1p(df['price'])
# After prediction: price_pred = np.expm1(model.predict(X_new))

print('Preprocessing pipeline defined. Fill in your column names above.')


---

# 🤖 Section 4: Choosing & Building Your Price Prediction Model
#### *The model is your decision — justify it*

---

You choose the model. There is no single correct answer.  
The requirement is that you **try at least 3 models**, compare them honestly,  
and justify your final choice.

### Candidate Models (ordered by typical performance on housing data)

| Model | Notes | Good Starting Point? |
|-------|-------|---------------------|
| **Linear Regression** | Fast, interpretable, weak on non-linear patterns | Yes — always build a baseline |
| **Ridge / Lasso** | Linear with regularisation — handles correlated features | Yes |
| **Random Forest Regressor** | Handles non-linearity, robust, good baseline | Yes |
| **Gradient Boosting (XGBoost/LightGBM)** | Usually best on tabular housing data | Strongly recommended |
| **Neural Network (PyTorch/Keras)** | Can be powerful with enough data and tuning | Advanced — optional |

### What to Report for Each Model
- MAE (Mean Absolute Error) — in original price units, most interpretable
- RMSE (Root Mean Squared Error) — penalises large errors more
- R² score — proportion of variance explained (1.0 = perfect)
- Training time
- A residual plot — predicted vs actual, coloured by error magnitude

### Key Decision: Raw Price vs Log Price

Try both:
```python
# Option A: predict raw price
y = df['price']
pred = model.predict(X_test)

# Option B: predict log price (often better for right-skewed distributions)
y = np.log1p(df['price'])
pred = np.expm1(model.predict(X_test))   # back to original scale
```
Report which gives better MAE on the original price scale.


In [ ]:
# ── Model comparison scaffold ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Split your data
# X = preprocessor.fit_transform(df[numeric_features + categorical_features])
# y = np.log1p(df['price'])   # log-transform recommended
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    pred_log = model.predict(X_te)
    pred     = np.expm1(pred_log)   # back to original price
    true     = np.expm1(y_te)
    mae  = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    r2   = r2_score(true, pred)
    print(f'{name:<30} MAE: ${mae:>10,.0f}  RMSE: ${rmse:>10,.0f}  R2: {r2:.4f}')
    return model

# Example usage (uncomment when your data is ready):
# models = [
#     ('Linear Regression',   LinearRegression()),
#     ('Ridge (alpha=1)',     Ridge(alpha=1.0)),
#     ('Random Forest',       RandomForestRegressor(n_estimators=200, random_state=42)),
#     ('Gradient Boosting',   GradientBoostingRegressor(n_estimators=300, random_state=42)),
# ]
# print(f'{"Model":<30} {"MAE":>15} {"RMSE":>15} {"R2":>8}')
# print('-'*72)
# trained_models = {}
# for name, m in models:
#     trained_models[name] = evaluate_model(name, m, X_train, X_test, y_train, y_test)

print('Scaffold ready. Load your data and uncomment the model comparison loop.')


In [ ]:
# ── Residual analysis — understanding where your model fails ─────────────────
# Run this after training your best model

# best_model = trained_models['Gradient Boosting']  # or whichever won
# pred_log   = best_model.predict(X_test)
# pred_price = np.expm1(pred_log)
# true_price = np.expm1(y_test)
# residuals  = true_price - pred_price

# fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# axes[0].scatter(true_price, pred_price, alpha=0.4, color='#0D7377', s=20)
# max_val = max(true_price.max(), pred_price.max())
# axes[0].plot([0,max_val],[0,max_val],'r--', label='Perfect prediction')
# axes[0].set_xlabel('Actual Price'); axes[0].set_ylabel('Predicted Price')
# axes[0].set_title('Predicted vs Actual', fontweight='bold'); axes[0].legend()

# axes[1].scatter(pred_price, residuals, alpha=0.4, color='#F0A500', s=20)
# axes[1].axhline(0, color='red', linestyle='--')
# axes[1].set_xlabel('Predicted Price'); axes[1].set_ylabel('Residual (Actual - Predicted)')
# axes[1].set_title('Residual Plot', fontweight='bold')

# axes[2].hist(residuals, bins=50, color='#0D7377', edgecolor='white')
# axes[2].axvline(0, color='red', linestyle='--')
# axes[2].set_xlabel('Residual'); axes[2].set_title('Residual Distribution', fontweight='bold')

# plt.tight_layout(); plt.show()
print('Uncomment the residual analysis after training your best model.')


---

# 🔍 Section 5: Building the House Recommender
#### *From price prediction to personalised recommendations*

---

The recommender answers: *'Given the house you described, here are similar properties  
you might also like — that are actually available in our dataset.'*

### The Approach: Content-Based Filtering with Similarity Search

When a user inputs house features to get a price prediction, those inputs define  
a **query vector** — a point in feature space. The recommender finds the K houses  
in your dataset whose feature vectors are most similar to that query.

```
User inputs features → query vector
       ↓
Compute distance to every house in dataset
       ↓
Return K closest houses (with their details and predicted prices)
```

### Similarity Metrics — Choose One

| Metric | Best When |
|--------|----------|
| **Euclidean distance** | All features are on similar scales (use after scaling) |
| **Cosine similarity** | Feature magnitudes vary, direction matters more |
| **Manhattan distance** | More robust to outlier dimensions |
| **Weighted distance** | You want some features (location, size) to matter more |

### Implementation Options

**Option A — KNN (simplest):**
```python
from sklearn.neighbors import NearestNeighbors
knn_rec = NearestNeighbors(n_neighbors=6, metric='euclidean')
knn_rec.fit(X_scaled_all)   # fit on ALL your data (preprocessed)
distances, indices = knn_rec.kneighbors(query_vector)
recommended = df.iloc[indices[0][1:]]  # skip index 0 if query is in dataset
```

**Option B — Faiss (faster for large datasets):**
```python
import faiss
index = faiss.IndexFlatL2(X_scaled_all.shape[1])
index.add(X_scaled_all.astype('float32'))
D, I = index.search(query_vector.astype('float32'), k=6)
recommended = df.iloc[I[0]]
```

**Option C — Weighted similarity (most sophisticated):**
Weight features by their importance from your price prediction model:
```python
# Use feature importances from Random Forest or XGBoost
importances = best_model.feature_importances_
# Multiply features by their importance before computing distances
X_weighted = X_scaled_all * importances
```

### What the Recommender Returns

For each recommended house, show:
- All key features (bedrooms, bathrooms, size, location, etc.)
- Actual price from dataset (or predicted price if no label)
- Similarity score (distance — lower is more similar)
- A brief comparison to the query (e.g. '2 more bedrooms, 15% larger')


In [ ]:
# ── Recommender scaffold ─────────────────────────────────────────────────────
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

class HouseRecommender:
    """
    Content-based house recommender using KNN similarity search.
    """
    def __init__(self, n_neighbors=6, metric='euclidean'):
        self.n_neighbors = n_neighbors
        self.metric      = metric
        self.knn         = NearestNeighbors(n_neighbors=n_neighbors, metric=metric)
        self.scaler      = StandardScaler()
        self.df_ref      = None
        self.feature_cols = None

    def fit(self, df, feature_cols):
        """
        Fit the recommender on your housing dataset.
        df:           full housing DataFrame
        feature_cols: list of columns to use for similarity
        """
        self.df_ref       = df.reset_index(drop=True)
        self.feature_cols = feature_cols
        X = df[feature_cols].fillna(df[feature_cols].median(numeric_only=True))
        X_scaled = self.scaler.fit_transform(X)
        self.knn.fit(X_scaled)
        return self

    def recommend(self, query_features: dict, k=5):
        """
        query_features: dict with feature_col → value for the user's house
        Returns: DataFrame of k most similar houses
        """
        query_row = pd.DataFrame([query_features])[self.feature_cols]
        query_row = query_row.fillna(self.df_ref[self.feature_cols].median(numeric_only=True))
        query_scaled = self.scaler.transform(query_row)
        distances, indices = self.knn.kneighbors(query_scaled, n_neighbors=k+1)
        # Return top-k matches (skip index 0 in case query is in dataset)
        recs = self.df_ref.iloc[indices[0][1:k+1]].copy()
        recs['similarity_score'] = distances[0][1:k+1]
        return recs.sort_values('similarity_score')

# Usage example (uncomment when your data is ready):
# feature_cols = ['bedrooms','bathrooms','sqm','age','neighbourhood_avg_price']
# recommender  = HouseRecommender(n_neighbors=6).fit(df, feature_cols)
# query = {'bedrooms':3,'bathrooms':2,'sqm':150,'age':10,'neighbourhood_avg_price':450000}
# recommendations = recommender.recommend(query, k=5)
# print(recommendations)
print('HouseRecommender class defined. Fill in your feature columns and test it.')


---

# 🌐 Section 6: Streamlit Deployment
#### *Building the web application that users interact with*

---

### What is Streamlit?

Streamlit is a Python library that turns scripts into interactive web apps  
with no frontend knowledge required. You write Python — it becomes a web app.

```bash
pip install streamlit
streamlit run app.py   # launches browser automatically
```

### Your App Architecture

```
app.py
├── Header / branding
├── Sidebar: user inputs (sliders, dropdowns, number inputs)
├── Section 1: Price Prediction
│   ├── Predicted price (large, prominent)
│   ├── Confidence range (e.g. ±10%)
│   └── Feature importance chart
└── Section 2: Similar Houses
    ├── Table of recommended properties
    ├── Comparison to user input
    └── Optional: map visualisation (if you have lat/lon)
```

### Key Streamlit Components You Will Use

| Component | Code | Use |
|-----------|------|-----|
| Title | `st.title('House Finder')` | App header |
| Slider | `st.slider('Bedrooms', 1, 10, 3)` | Numeric input with range |
| Selectbox | `st.selectbox('Type', ['Apartment','House'])` | Dropdown |
| Number input | `st.number_input('Size (sqm)', 20, 2000, 100)` | Precise number |
| Button | `st.button('Predict Price')` | Trigger action |
| Metric | `st.metric('Estimated Price', '$450,000')` | Highlighted number |
| DataFrame | `st.dataframe(recs_df)` | Table display |
| Chart | `st.bar_chart(feature_importances)` | Quick chart |
| Columns | `col1, col2 = st.columns(2)` | Side-by-side layout |
| Spinner | `with st.spinner('Predicting...'):` | Loading indicator |
| Success | `st.success('Prediction ready!')` | Status message |


In [ ]:
# ── app.py — Your Streamlit app (save this as app.py and run with streamlit run app.py)
# This is a scaffold — you will need to fill in your actual model and data paths

streamlit_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

# ── Page config ─────────────────────────────────────────────────────────────
st.set_page_config(
    page_title   = 'House Price Predictor',
    page_icon    = '🏠',
    layout       = 'wide',
    initial_sidebar_state = 'expanded'
)

# ── Load models (cached for performance) ────────────────────────────────────
@st.cache_resource
def load_models():
    model        = joblib.load('price_model.pkl')       # your trained model
    preprocessor = joblib.load('preprocessor.pkl')      # your fitted pipeline
    recommender  = joblib.load('recommender.pkl')       # your HouseRecommender
    df           = pd.read_csv('housing_data.csv')      # your full dataset
    return model, preprocessor, recommender, df

model, preprocessor, recommender, df = load_models()

# ── Header ──────────────────────────────────────────────────────────────────
st.title('🏠 House Price Predictor & Recommender')
st.markdown('Enter the details of a property to get an instant price estimate and discover similar listings.')
st.divider()

# ── Sidebar: User Inputs ─────────────────────────────────────────────────────
st.sidebar.header('Property Features')
st.sidebar.markdown('Describe the property you are interested in:')

bedrooms       = st.sidebar.slider('Bedrooms',         1, 10, 3)
bathrooms      = st.sidebar.slider('Bathrooms',        1, 8,  2)
size_sqm       = st.sidebar.number_input('Total Area (sqm)', 20, 2000, 120)
property_age   = st.sidebar.slider('Property Age (years)', 0, 100, 10)
property_type  = st.sidebar.selectbox('Property Type', ['Apartment','Detached House','Semi-Detached','Townhouse','Villa'])
neighbourhood  = st.sidebar.selectbox('Neighbourhood', sorted(df['neighbourhood'].unique()))  # update column name
has_garage     = st.sidebar.checkbox('Has Garage', value=True)
has_garden     = st.sidebar.checkbox('Has Garden', value=False)
condition      = st.sidebar.selectbox('Condition', ['Excellent','Good','Fair','Needs Work'])

predict_btn = st.sidebar.button('🔍 Predict Price & Find Similar', type='primary')

# ── Main content ─────────────────────────────────────────────────────────────
if predict_btn:
    with st.spinner('Analysing property...'):
        # Build query dict
        query = {
            'bedrooms':      bedrooms,
            'bathrooms':     bathrooms,
            'size_sqm':      size_sqm,
            'property_age':  property_age,
            'property_type': property_type,
            'neighbourhood': neighbourhood,
            'has_garage':    int(has_garage),
            'has_garden':    int(has_garden),
            'condition':     condition,
        }
        query_df   = pd.DataFrame([query])
        X_query    = preprocessor.transform(query_df)
        log_pred   = model.predict(X_query)[0]
        pred_price = np.expm1(log_pred)                     # back to original scale
        low_price  = pred_price * 0.90                      # confidence range
        high_price = pred_price * 1.10

    # ── Price Prediction Section ─────────────────────────────────────────────
    st.subheader('💰 Price Estimate')
    col1, col2, col3 = st.columns(3)
    col1.metric('Low Estimate',  f'${low_price:,.0f}')
    col2.metric('Predicted Price', f'${pred_price:,.0f}', delta=None)
    col3.metric('High Estimate', f'${high_price:,.0f}')
    st.success(f'Estimated market value: **${pred_price:,.0f}** (±10% confidence range)')
    st.divider()

    # ── Feature Importance ───────────────────────────────────────────────────
    if hasattr(model, "feature_importances_"):
        st.subheader('📊 What Drives the Price')
        # Get feature names from preprocessor
        try:
            feature_names = preprocessor.get_feature_names_out()
            imp_df = pd.Series(model.feature_importances_, index=feature_names)
            imp_df = imp_df.sort_values(ascending=True).tail(10)
            fig, ax = plt.subplots(figsize=(8, 4))
            imp_df.plot(kind="barh", ax=ax, color="#0D7377", edgecolor="white")
            ax.set_title("Top 10 Most Important Features", fontweight="bold")
            ax.set_xlabel("Importance")
            st.pyplot(fig)
        except Exception:
            pass
    st.divider()

    # ── Recommendations ──────────────────────────────────────────────────────
    st.subheader('🔍 Similar Properties You Might Like')
    recs = recommender.recommend(query, k=5)

    for i, (_, row) in enumerate(recs.iterrows(), 1):
        with st.expander(f"Property {i} — ${row.get('price',0):,.0f}  |  Similarity: {1/(1+row['similarity_score']):.0%}"):
            col_a, col_b = st.columns(2)
            col_a.write(f"**Bedrooms:** {row.get('bedrooms', 'N/A')}")
            col_a.write(f"**Bathrooms:** {row.get('bathrooms', 'N/A')}")
            col_a.write(f"**Size:** {row.get('size_sqm', 'N/A')} sqm")
            col_b.write(f"**Location:** {row.get('neighbourhood', 'N/A')}")
            col_b.write(f"**Type:** {row.get('property_type', 'N/A')}")
            col_b.write(f"**Age:** {row.get('property_age', 'N/A')} years")

else:
    st.info('👈 Enter property details in the sidebar and click **Predict Price & Find Similar** to get started.')
    st.image('https://images.unsplash.com/photo-1570129477492-45c003edd2be?w=1200', use_column_width=True)
'''

# Save app.py
with open('app.py', 'w') as f:
    f.write(streamlit_code.strip())

print('app.py written successfully.')
print('To run: streamlit run app.py')
print('To deploy publicly: push to GitHub + connect to share.streamlit.io (free)')


---

# ☁️ Section 7: Saving Models & Deploying to Streamlit Cloud
#### *From notebook to live public URL*

---

### Step 1 — Save Your Trained Artefacts

Before deploying, save everything the app needs:
```python
import joblib

joblib.dump(best_model,    'price_model.pkl')
joblib.dump(preprocessor,  'preprocessor.pkl')
joblib.dump(recommender,   'recommender.pkl')
df.to_csv('housing_data.csv', index=False)
```

### Step 2 — Project Structure

Your project folder should look like this:
```
house-predictor/
├── app.py                  ← Streamlit application
├── price_model.pkl         ← trained regression model
├── preprocessor.pkl        ← fitted sklearn Pipeline
├── recommender.pkl         ← fitted HouseRecommender
├── housing_data.csv        ← your dataset (for recommendations)
├── requirements.txt        ← Python dependencies
└── README.md               ← project description
```

### Step 3 — requirements.txt

```
streamlit>=1.28
pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
matplotlib>=3.7
seaborn>=0.12
joblib>=1.3
xgboost>=2.0          # if you used XGBoost
lightgbm>=4.0         # if you used LightGBM
plotly>=5.0           # if you added interactive charts
```

### Step 4 — Deploy to Streamlit Community Cloud (Free)

1. Push your project folder to a **public GitHub repository**
2. Go to [share.streamlit.io](https://share.streamlit.io)
3. Sign in with GitHub
4. Click **'New app'** → select your repo → select `app.py`
5. Click **Deploy** — your app gets a public URL in ~2 minutes

Your app is now live. Share the link in your portfolio, LinkedIn, and CV.

### Step 5 — Optional: Add a Map

If your dataset has latitude/longitude:
```python
import plotly.express as px

fig = px.scatter_mapbox(
    recs,
    lat='latitude', lon='longitude',
    size='price', color='price',
    hover_data=['bedrooms','bathrooms','size_sqm'],
    zoom=11, mapbox_style='open-street-map'
)
st.plotly_chart(fig, use_container_width=True)
```


In [ ]:
# ── Save your artefacts — run this after training ─────────────────────────────
import joblib

# Uncomment and run after your models are trained:
# joblib.dump(best_model,    'price_model.pkl')
# joblib.dump(preprocessor,  'preprocessor.pkl')
# joblib.dump(recommender,   'recommender.pkl')
# df.to_csv('housing_data.csv', index=False)
# print('All artefacts saved.')

# requirements.txt content
requirements = """streamlit>=1.28
pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
matplotlib>=3.7
seaborn>=0.12
joblib>=1.3
xgboost>=2.0
plotly>=5.0
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)
print('requirements.txt written.')

readme = """# House Price Predictor & Recommender

An ML-powered web application that predicts house prices and recommends similar properties.

## Features
- **Price Prediction**: Regression model trained on [your dataset name]
- **House Recommender**: Content-based similarity search using KNN
- **Interactive UI**: Built with Streamlit

## Tech Stack
Python · scikit-learn · XGBoost · Streamlit · Pandas · NumPy

## Run Locally
```bash
pip install -r requirements.txt
streamlit run app.py
```

## Live Demo
[Add your Streamlit Cloud URL here after deployment]
"""

with open('README.md', 'w') as f:
    f.write(readme)
print('README.md written.')
print('\nProject is ready to push to GitHub and deploy on Streamlit Cloud.')


---

# 📋 Section 8: Evaluation Rubric & Submission
#### *How your project will be assessed*

---

### Grading Rubric

| Component | Marks | What We Look For |
|-----------|-------|------------------|
| **Data sourcing & description** | 10 | Justified choice, dataset documented, stats reported |
| **EDA quality** | 15 | Minimum 5 charts, all interpreted, findings acted on |
| **Data cleaning** | 10 | All decisions justified, before/after reported |
| **Feature engineering** | 15 | At least 4 new features, each with a rationale |
| **Price prediction model** | 20 | ≥3 models compared, best chosen with justification |
| **House recommender** | 15 | Works correctly, returns relevant results |
| **Streamlit app** | 10 | Functional, clean UI, both systems integrated |
| **Deployment** | 5 | Live public URL shared |
| **Total** | **100** | |

### Submission Requirements

Submit the following:
1. **Jupyter Notebook** — your full analysis, cleaning, engineering, and modelling
2. **GitHub repository URL** — containing app.py, requirements.txt, README.md
3. **Streamlit Cloud URL** — the live deployed application
4. **3-minute verbal presentation** — walk through your data choice, model decision, and one key insight

### Tips for an Outstanding Project

- **Use local market data** — a Lagos or Nairobi housing dataset is far more interesting and  
  impressive than the Ames, Iowa dataset everyone uses
- **Narrate your decisions** — write Markdown cells explaining why you made each choice,  
  not just what you did
- **Handle edge cases in the app** — what happens if the user enters 0 bedrooms?  
  What if the neighbourhood they pick has no similar houses?
- **Add a comparison table** — show how the recommended houses differ from the query
- **Design matters** — a well-styled Streamlit app with consistent colours and clear labels  
  signals professionalism
- **Write the README as if a hiring manager is reading it** — they often are

---

> 🔑 **The best projects are not the ones with the highest accuracy.**  
> They are the ones that clearly solve a real problem, explain their reasoning,  
> and produce something a real person would actually use.


---
## ✅ Project Checklist

Work through this before submission:

- [ ] Dataset sourced, described, and documented
- [ ] EDA complete — minimum 5 charts with interpretations
- [ ] Data cleaned — all issues addressed and justified
- [ ] Feature engineering — at least 4 new features created
- [ ] Preprocessing pipeline built (handles missing values, encoding, scaling)
- [ ] At least 3 regression models trained and compared
- [ ] Best model selected with justification
- [ ] Residual analysis done — you know where your model fails
- [ ] HouseRecommender fitted and returning sensible results
- [ ] app.py written and tested locally
- [ ] All artefacts saved (model, preprocessor, recommender, data)
- [ ] requirements.txt and README.md written
- [ ] GitHub repository created and code pushed
- [ ] App deployed on Streamlit Cloud — public URL working
- [ ] Notebook is clean, well-commented, and tells a story

<div style='background:linear-gradient(135deg,#1A2E4A 0%,#0D7377 100%);padding:35px;border-radius:10px;color:white;text-align:center;'>
  <h3 style='margin:0 0 10px 0;color:#14BDBD;'>Ship it.</h3>
  <p style='color:#D0D7E3;margin:0 0 10px 0;'>A deployed, working application with your name on it is worth  
more than a hundred perfect tutorial completions.</p>
  <p style='color:#F0A500;font-weight:bold;margin:0;'>This is your portfolio. Build something you are proud to show. 🚀</p>
</div>
